In [0]:
dbutils.widgets.text("env","","Enter the environment in lower case")
env = dbutils.widgets.get("env")

In [0]:
%run "./commons"

## Reading from bronze raw_roads 

In [0]:
def read_BronzeRoadsTable(environment):
    print('Reading the bronzse table raw_roads data: ',end='')
    df_bronzeRoads= (spark.readStream
                    .table(f"`{environment}_catalog`.`bronze`.`raw_roads`"))
    print(f"Reading {environment}_catalog.bronze.raw_roads Success")
    print("***************")
    return df_bronzeRoads

## Creating road_Cateogry_name column

In [0]:
def road_Category(df):
    print('Creating Road Category Name Column: ', end='')
    from pyspark.sql.functions import when, col

    df_road_cat = df.withColumn("Road_Category_NAme",
                when(col('Road_Category')=='TA','Class A Trunk Road')
                .when(col('Road_Category')=='TM','Class A Trunk Motor')
                 .when(col('Road_Category')=='PA','Class A Principal road')
                  .when(col('Road_Category')=='PM','Class A Principal Motorway')
                   .when(col('Road_Category')=='M','Class B road')             
                   .otherwise('NA')       
                                )
    print('Success')
    print('****************')
    return df_road_cat

## Creating road_type column

In [0]:
def road_Type(df):
    print('Creating Road Type NAme Column: ', end='')
    from pyspark.sql.functions import when, col

    df_road_Type= df.withColumn("Road_Type",
                when(col('Road_Category_Name').like('%Class A%'),'Major')
                .when(col('Road_Category_Name').like('%Class B%'),'Minor')
                .otherwise('NA')             
                 )
    print('Success')
    print('*****************************')
    return df_road_Type

## Writing data to silver_roads in silver schema

In [0]:
def write_Roads_SilverTable(StreamingDF, environment):
    print('Writing the silver_roads Data: ', end='')

    write_StreamSilver_R= (StreamingDF.writeStream
                .format("delta")
                .option("checkpointLocation", checkpoint + "/SilverRoadsLoad/Checkpt/")
                .outputMode('append')
                .queryName('SilverRoadsWriteStream')
                .trigger(availableNow=True)
                .toTable(F"`{environment}_catalog`.`silver`.`silver_roads`"))
    
    write_StreamSilver_R.awaitTermination()
    print(f'Writing `{environment}_catalog`.`silver`.`silver_roads` Success')

## Calling Functions

In [0]:
## Reading the bronze traffic data
df_roads = read_BronzeRoadsTable(env)

# to remove duplicate rows
df_noDups = remove_Dups(df_roads)

# to replace any nulls values
AllColumns = df_noDups.schema.names
df_clean = handle_NULLS(df_noDups , AllColumns)

## Creating Road Category_name
df_roadCat = road_Category(df_clean)

## Creating Road_Type column
df_Type = road_Type(df_roadCat)

## Writing data to silver_roads table
write_Roads_SilverTable(df_Type, env)